In [4]:
# Imports
import pandas as pd
import re
import json
import duckdb
import matplotlib.pyplot as plt

## Get ground Truth

In [5]:
ground_truth_napoleon = pd.read_csv('../data/weight_files/ground_truth_raw/GT_napoleon_in_edges_raw.csv')

In [43]:
ground_truth_napoleon = ground_truth_napoleon.drop_duplicates(subset='id', keep='first').reset_index(drop=True)

In [44]:
ground_truth_napoleon = ground_truth_napoleon[["label", "text" ]]

In [45]:
print(ground_truth_napoleon.shape)
ground_truth_napoleon

(46, 2)


,label,text
0,"[{""end"":56226,""text"":""Napoleon"",""start"":56218,...",upload/286914/1fdf5af5-Voltaire_en.txt
1,"[{""end"":929,""text"":""Napoleon"",""start"":921,""lab...",upload/286914/5cc5cfef-Vincent-Marie_Viénot_Co...
2,"[{""end"":38625,""text"":""Napoleon Bonaparte"",""sta...",upload/286914/341b596e-Robert_Clive_en.txt
3,NaN,upload/286914/6b10b085-Prince_Philippe_Count_o...
4,NaN,upload/286914/73f76387-Prince_Josias_of_Saxe-C...
5,"[{""end"":5885,""text"":""Napoleon's"",""start"":5875,...",upload/286914/d9f53915-Prince_Frederick_Duke_o...
6,"[{""end"":9272,""text"":""Napoleon"",""start"":9264,""l...",upload/286914/3d352a8f-Prince_Ferdinand_Philip...
7,"[{""end"":6466,""text"":""Napoleon"",""start"":6458,""l...",upload/286914/f797fa95-Paolo_Virzì_en.txt
8,"[{""end"":868,""text"":""Napoleon Bonaparte's"",""sta...",upload/286914/57a687d9-Oscar_I_of_Sweden_en.txt
9,"[{""end"":627,""text"":""Napoleon"",""start"":619,""lab...",upload/286914/645a6703-Nicolas-Charles_Bochsa_...


In [46]:
def extract_name(path):
    name = path.split('/')[-1]                  # 341b596e-Robert_Clive_en.txt
    name = re.sub(r'^[0-9a-f]+-', '', name)      # drop hash prefix -> Robert_Clive_en.txt
    name = re.sub(r'\.[^.]+$', '', name)         # drop extension -> Robert_Clive_en
    name = re.sub(r'_[a-z]{2}$', '', name)       # drop trailing language code -> Robert_Clive
    return name.replace('_', ' ')                # -> Robert Clive

ground_truth_napoleon['name'] = ground_truth_napoleon['text'].apply(extract_name)

In [47]:
ground_truth_napoleon

,label,text,name
0,"[{""end"":56226,""text"":""Napoleon"",""start"":56218,...",upload/286914/1fdf5af5-Voltaire_en.txt,Voltaire
1,"[{""end"":929,""text"":""Napoleon"",""start"":921,""lab...",upload/286914/5cc5cfef-Vincent-Marie_Viénot_Co...,Vincent-Marie Viénot Count of Vaublanc
2,"[{""end"":38625,""text"":""Napoleon Bonaparte"",""sta...",upload/286914/341b596e-Robert_Clive_en.txt,Robert Clive
3,NaN,upload/286914/6b10b085-Prince_Philippe_Count_o...,Prince Philippe Count of Paris
4,NaN,upload/286914/73f76387-Prince_Josias_of_Saxe-C...,Prince Josias of Saxe-Coburg-Saalfeld
5,"[{""end"":5885,""text"":""Napoleon's"",""start"":5875,...",upload/286914/d9f53915-Prince_Frederick_Duke_o...,Prince Frederick Duke of York and Albany
6,"[{""end"":9272,""text"":""Napoleon"",""start"":9264,""l...",upload/286914/3d352a8f-Prince_Ferdinand_Philip...,Prince Ferdinand Philippe Duke of Orléans
7,"[{""end"":6466,""text"":""Napoleon"",""start"":6458,""l...",upload/286914/f797fa95-Paolo_Virzì_en.txt,Paolo Virzì
8,"[{""end"":868,""text"":""Napoleon Bonaparte's"",""sta...",upload/286914/57a687d9-Oscar_I_of_Sweden_en.txt,Oscar I of Sweden
9,"[{""end"":627,""text"":""Napoleon"",""start"":619,""lab...",upload/286914/645a6703-Nicolas-Charles_Bochsa_...,Nicolas-Charles Bochsa


In [48]:
con = duckdb.connect('data/out/napo_sources.db')
title_to_id = con.execute("""
    SELECT 'Q' || wp.wikidata_id, wp.page_title
    FROM wiki_page wp
    JOIN extract e ON e.wiki_page_id = wp.wiki_page_id
""").fetchall()

con.close()

In [49]:
title_to_id

[('Q63023', 'Friedrich Wilhelm Freiherr von Bülow'),
 ('Q607190', 'Paolo Virzì'),
 ('Q451608', 'Gaspard de Prony'),
 ('Q334973', 'Félicité de La Mennais'),
 ('Q115995', "Achille Baraguey d'Hilliers"),
 ('Q360481', 'Vincent-Marie Viénot, Count of Vaublanc'),
 ('Q7243', 'Leo Tolstoy'),
 ('Q57511', 'Franz Joseph Gall'),
 ('Q78873', 'Ida Laura Pfeiffer'),
 ('Q7721', 'Napoleon III'),
 ('Q266339', 'Józef Bem'),
 ('Q310807', 'Archduke Albrecht, Duke of Teschen'),
 ('Q192615', 'Ferdinand Foch'),
 ('Q456812', 'Donald Adamson'),
 ('Q356933', 'François Christophe de Kellermann'),
 ('Q723247', "Maurice d'Elbée"),
 ('Q665957', 'Jacques Laffitte'),
 ('Q5547', 'Antonio Canova'),
 ('Q52926', 'Oscar I of Sweden'),
 ('Q162296', 'Robert Clive'),
 ('Q580516', 'Jean Maximilien Lamarque'),
 ('Q314917', "Daniel O'Connell"),
 ('Q555304', 'Georges Cadoudal'),
 ('Q142141', 'Mikhail Tukhachevsky'),
 ('Q354477', 'Prince Philippe, Count of Paris'),
 ('Q983354', 'Germinal Pierre Dandelin'),
 ('Q7349', 'Joseph Haydn

In [50]:
# build a lookup dict from the tuple list (Qid, name) -> {name: Qid}
name_to_qid = {name: qid for qid, name in title_to_id}
name_to_qid = {k.replace(',', '').replace("'", ''): v for k, v in name_to_qid.items()}
ground_truth_napoleon['ID'] = ground_truth_napoleon['name'].map(name_to_qid)


In [51]:
ground_truth_napoleon['label'] = ground_truth_napoleon['label'].apply(
    lambda x: json.loads(x) if pd.notna(x) else x
)

In [52]:
def get_weight(x):
    if isinstance(x, list):
        return len(x)
    return 0

ground_truth_napoleon['GT_weight'] = ground_truth_napoleon['label'].apply(get_weight)

In [53]:
ground_truth_napoleon = ground_truth_napoleon.drop(['label', 'text'], axis=1)

In [54]:
ground_truth_napoleon

,name,ID,GT_weight
0,Voltaire,Q9068,1
1,Vincent-Marie Viénot Count of Vaublanc,Q360481,13
2,Robert Clive,Q162296,2
3,Prince Philippe Count of Paris,Q354477,0
4,Prince Josias of Saxe-Coburg-Saalfeld,Q68771,0
5,Prince Frederick Duke of York and Albany,Q213573,1
6,Prince Ferdinand Philippe Duke of Orléans,Q458688,2
7,Paolo Virzì,Q607190,1
8,Oscar I of Sweden,Q52926,3
9,Nicolas-Charles Bochsa,Q251397,1


In [ ]:
ground_truth_napoleon.to_csv('../data/weight_files/GT_weights.csv', index=False)